# 00 — Setup & Environment Check

**puhekieli-llm — teaching a small model to translate English → *spoken* Finnish**

Off-the-shelf models translate EN→FI into formal written Finnish (**kirjakieli**).
The goal of this personal holiday project is the register people actually *speak*:
**puhekieli**. This notebook is the warm-up. By the end you'll have confirmed:
1. The Python env + ML stack is installed
2. PyTorch can use **Apple Metal (MPS)** for GPU acceleration
3. The project paths & task config are wired up

> Everything runs *locally* on this machine. It's an experiment — let's see how far
> we get.

In [ ]:
import torch, platform, sys
from puhekieli_llm.config import summary, DEVICE, SRC_LANG, TGT_LANG, TGT_REGISTER

print('python :', platform.python_version())
print('torch  :', torch.__version__)
print('device :', DEVICE)
print('mps available:', torch.backends.mps.is_available())
print('task   :', f'{SRC_LANG} -> {TGT_LANG} ({TGT_REGISTER})')

## Quick MPS smoke test
A matmul on the GPU should be noticeably faster than CPU once tensors are large.

In [ ]:
import time

def bench(device, n=4096, iters=10):
    x = torch.randn(n, n, device=device)
    if device == 'mps':
        torch.mps.synchronize()
    t0 = time.time()
    for _ in range(iters):
        y = x @ x
    if device == 'mps':
        torch.mps.synchronize()
    return (time.time() - t0) / iters

cpu_t = bench('cpu')
mps_t = bench('mps')
print(f'cpu : {cpu_t*1000:7.1f} ms / matmul')
print(f'mps : {mps_t*1000:7.1f} ms / matmul')
print(f'speedup: {cpu_t/mps_t:.1f}x')

## The goal: puhekieli, not kirjakieli

Standard MT/LLMs default to formal written Finnish. We want everyday *spoken*
Finnish. Same meaning, very different surface form:

In [ ]:
from puhekieli_llm.config import SOURCES, active_sources

examples = [
    ('I am',    'minä olen',   'mä oon'),
    ('we go',   'me menemme',  'me mennään'),
    ('they',    'he',          'ne'),
    ("don't know", 'en tiedä', 'emmä tiiä'),
]
print(f"{'english':14s} {'kirjakieli':16s} {'puhekieli (want)'}")
for en, kirja, puhe in examples:
    print(f'  {en:12s} {kirja:16s} {puhe}')

print()
print(summary())
print()
print('Registered data sources (user picks these):')
if not SOURCES:
    print('  (none yet — add cleared sources to config.py::SOURCES)')
for name, meta in SOURCES.items():
    print(f"  {name:18s} register={meta.get('register'):10s} status={meta.get('status')}")

## ✅ If all cells ran, Phase 0 is done.

**Next:** decide your data sources (register them in `config.py::SOURCES`), then
`01_collect.ipynb` — gather spoken-Finnish / EN↔FI parallel data into `data/raw`.